# GCI Final Assignment - Industrial Incident Severity Intelligence

## 1. Introduction & Objectives
This notebook implements an exploratory data analysis (EDA), feature engineering pipeline, and baseline machine learning model to classify industrial incident severity levels (`severity_level`) and support preventive safety operations.

## 2. Setup & Environment Dependencies

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline
sns.set_theme(style="whitegrid")

## 3. Data Extraction from Warehouse / Database

In [ ]:
# Load cleaned data directly from SQLite database or raw dataset
db_path = "../database/incidents.db"
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM incidents", conn)
    conn.close()
else:
    df = pd.read_csv("../data/raw/incidents_dataset.csv")

print(f"Total records loaded: {len(df)}")
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print("=== Dataset Info ===")
print(df.info())

print("\n=== Severity Breakdown ===")
print(df['severity_level'].value_counts())

# Distribution Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='severity_level', ax=axes[0], palette='viridis')
axes[0].set_title('Incident Count by Severity Level')

sns.boxplot(data=df, x='severity_level', y='days_lost', ax=axes[1], palette='plasma')
axes[1].set_title('Days Lost vs. Severity Level')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# One-Hot Encoding for categorical features (sector)
X = pd.get_dummies(df[['sector', 'days_lost']], columns=['sector'], drop_first=True)
y = df['severity_level'].astype(int)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

## 6. Model Training (Random Forest Baseline)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("Model training complete.")

## 7. Model Evaluation & Performance Metrics

In [ ]:
y_pred = model.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Severity')
plt.ylabel('True Severity')
plt.show()

## 8. Business Conclusions & Limitations
- **Impact**: The model effectively highlights high-risk sectors and correlates operational days lost with severe incidents.
- **Limitations**: Dataset size needs expansion in production environments for hyperparameter tuning and deep learning evaluation.